# **Gender Pronouns - 1st Person Extraction**<br>

## **1.** OpenSub Dataset Upload

In [1]:
import sys, re, pickle, copy, os, types
import pandas as pd
from tqdm import tqdm as _tqdm
from modules.OpensubHandler import OpenSubDataset

In [2]:
root = "../local_data/raw/opus_opensub/en-pl.txt/"
files = ('OpenSubtitles.en-pl.en', 'OpenSubtitles.en-pl.pl')
NUM_LINES = 5000000  
ds = OpenSubDataset(root, files, NUM_LINES)
df = ds.clean()
df.head(3)

,eng_text,pol_text,eng_split,pol_split
0,You want to call your daddy?,Chcesz zadzwonić do taty?,"[you, want, to, call, your, daddy, ?]","[chcesz, zadzwonić, do, taty, ?]"
1,"Yeah, I want to tell him I'm okay.","Tak, powiem, że wszystko w porządku.","[yeah, ,, i, want, to, tell, him, i, 'm, okay, .]","[tak, ,, powiem, ,, że, wszystko, w, porządku, .]"
2,Okay.,Dobrze.,"[okay, .]","[dobrze, .]"


## **2.** Helper Functions

In [3]:
def search(snt, re_pattern):
    return bool(re.search(re_pattern, snt, re.IGNORECASE))

def apply_mask(df, re_pattern, col='pol_text'):
    return df[col].apply(lambda snt: search(snt, re_pattern))

def extract(df_main, df_female, df_male, mask_f, mask_m):
    assert len(df_main) - (~(mask_f | mask_m)).sum() == (mask_f | mask_m).sum()
    
    df_female = pd.concat([df_female, df_main[mask_f]], ignore_index=True)
    df_male = pd.concat([df_male, df_main[mask_m]], ignore_index=True)
    df_main = df_main[~(mask_f | mask_m)].reset_index(drop=True)
    return df_main, df_female, df_male

In [4]:
def report_counts(**counts):
    print(" | ".join(f"{k.replace('_', ' ')}: {v.sum() if hasattr(v, 'sum') else v}" for k, v in counts.items()))

def report_shapes(**dfs):
    print(" | ".join(f"{k}: {v.shape[0]}" for k, v in dfs.items()))

## **3.** First-Person Sentence Filter

In [5]:
df_text = df[['eng_text', 'pol_text']].copy()

mask_first_person = df_text['eng_text'].str.contains(r"\b(?:i['’](?:m|ve|d|ll)|i|me|my|mine|myself)\b",
                                                     case=False, na=False, regex=True)
print("Num examples: ", mask_first_person.sum(), '\n')

df_first_person = df_text[mask_first_person].reset_index(drop=True)
df_text = df_text[~mask_first_person].reset_index(drop=True)

df_first_person.sample(3)

Num examples:  907395 



,eng_text,pol_text
731502,I never editorialize.,Nigdy nie komentuję.
124257,I'm afraid you've got me wrong.,"Obawiam się, że źle mnie oceniasz."
272536,How would I know?,Skąd miałbym wiedzieć?


## **4.** Gender Extraction

In [6]:
df_female = pd.DataFrame(columns=df_first_person.columns)
df_male = pd.DataFrame(columns=df_first_person.columns)

### **GROUP 1**: Past tense endings (łam/łem | łabym/łbym)

In [7]:
edge_cases_f = apply_mask(df_first_person, r'\bdziałam\b|\bdziałam\b|\bzłam\b|\bodłam\b|\brozłam\b|\bs[lł]abym\b')
mask_f = (apply_mask(df_first_person, r'\b\w+łam\b|\b\w+[łl]abym\b') & ~edge_cases_f)

edge_cases_m = apply_mask(
    df_first_person,
    r'\b(materia|cia|z|ko|czo|gard|krzes|mas|źród|god|skrzyd|siod|dzie|szk|do|ty|anio|\
    tytu|artyku|kościo|zespo|szczegó|sto|udzia|wydzia|genera|admira|py|diab|oddzia|kana|kryszta|myd|zapa)łem\b')
mask_m = (apply_mask(df_first_person, r'\b\w+łem\b|\b\w+[łl]bym\b') & ~edge_cases_m)

print(f"Num edge case female: {edge_cases_f.sum()} | Num edge case male: {edge_cases_m.sum()}\n")
print(f"Num female: {mask_f.sum()} | Num male: {mask_m.sum()}")

df_first_person, df_female, df_male = extract(df_first_person, df_female, df_male, mask_f, mask_m)

Num edge case female: 78 | Num edge case male: 525

Num female: 48807 | Num male: 122013


In [8]:
print(f"{'='*60}\ndf_female sample:\n\n{df_female['pol_text'].sample(3)}\n{'='*60}")
print(f"df_male sample:\n\n{df_male['pol_text'].sample(3)}\n{'='*60}")

df_female sample:

4843                  Zapomniałam o Laurze.
10583                      Kiedyś znałam...
38478    Powiedziałam mu, że to niemożliwe.
Name: pol_text, dtype: object
df_male sample:

21569     Poszedłem do mojego przyjaciela, prokuratora g...
113635            Przyjechałem do Obelisku w dobrej wierze.
115472          Dałem ci srebro i złoto, skarby królewskie.
Name: pol_text, dtype: object


### **GROUP 2A**: Będę/[..]bym + participle

In [9]:
mask_f = apply_mask(df_first_person, r"\b\w*(bym|bom|będę)\b.+\b\w+ła\b")
mask_m = apply_mask(df_first_person, r"\b\w*(bym|bom|będę)\b.+\b\w+ł\b")
print(f"Num female: {mask_f.sum()} | Num male: {mask_m.sum()}")
df_first_person, df_female, df_male = extract(df_first_person, df_female, df_male, mask_f, mask_m)

Num female: 4246 | Num male: 13153


### **GROUP 2B**: Będę/być/[..]bym + participle

In [10]:
edge_cases_f = apply_mask(df_first_person, r'\b\w*(bym|będę|być)\b\s+\b(sług|gwiazd|glin|inn|sierot|mężczyzn|patriot|maszyn|idiot|\
                                             maszyn|sędzi|tat|sob|osob|azjat|kalek|niemow|głow|porażk|ręk|gaduł|moj|mog|mn|biał|swoj|przynęt)ą\b')

mask_f = apply_mask(df_first_person, r"\b\w*(bym|być|będę)\b.+\b(?!potrzebn|.*pana\b)\w+na\b") | \
         apply_mask(df_first_person, r"\b\w*(bym|być|będę)\b\s+\b\w+ła\b") | \
         apply_mask(df_first_person, r"\b\w*(bym|być|będę)\b\s+\b(?!wa)\w+sza\b") | \
         (apply_mask(df_first_person, r'\b\w*(bym|będę|być)\b\s+\b\w+([ią]c|(os|ie)t|[wkłjn])ą\b') & ~edge_cases_f)

edge_cases_m = apply_mask(df_first_person, r"\b\w*(bym|być|będę)\b.+\bczłowiekiem\b")

mask_m = (apply_mask(df_first_person, r"\b\w*(bym|bom|będę)\b.+\b\w+((n|w|t|ł|sz)y|ym|wym)\b") | \
         apply_mask(df_first_person, r"\b\w*(bym|być|będę)\b.+\b\w*(mężczyzną|kaleką)\b")) & ~edge_cases_m

print(f"Num female: {mask_f.sum()} | Num male: {mask_m.sum()}")
df_first_person, df_female, df_male = extract(df_first_person, df_female, df_male, mask_f, mask_m)

Num female: 1168 | Num male: 1404


### **GROUP 3A**: Jestem... Manual Profession Conversion

In [11]:
def get_female_nouns(df_main, nouns_dict, suffixes_dct):
    nouns_pattern = r'|'.join(re.escape(key) for key in nouns_dict.keys())
    mask_nouns = apply_mask(df_main, rf'\bjestem\b.+\b({nouns_pattern})\b')
    print(f"Num examples: {mask_nouns.sum()}")
    female_nouns = df_main[mask_nouns].reset_index(drop=True)
    dct_comb = nouns_dict | suffixes_dct
    pattern_comb = r'|'.join(re.escape(key) for key in dct_comb.keys())
    female_nouns['pol_text'] = female_nouns['pol_text'].apply(lambda snt: re.sub(pattern_comb, 
                                                                                 lambda match: dct_comb[match.group(0).lower()], 
                                                                                 snt, flags=re.IGNORECASE))
    return female_nouns, mask_nouns

- ### **1.** ...arzem --> ...arką (e.g. lekarzem-lekarką | pisarzem-pisarką)

In [12]:
pattern_1 = r'\bjestem\b.+\b(\w+arzem)\b'
nouns_male = df_first_person['pol_text'].str.extract(pattern_1, flags=re.IGNORECASE)[0].str.lower().value_counts()
nouns_male = nouns_male[nouns_male >= 5].to_dict()
print(nouns_male)

{'lekarzem': 215, 'pisarzem': 36, 'dziennikarzem': 27, 'kucharzem': 19, 'szczęściarzem': 18, 'sekretarzem': 17, 'gospodarzem': 16, 'malarzem': 14, 'weterynarzem': 13, 'komisarzem': 12, 'ochroniarzem': 12, 'marynarzem': 10, 'handlarzem': 9, 'gliniarzem': 8, 'żeglarzem': 7, 'nudziarzem': 6, 'piosenkarzem': 5, 'taksówkarzem': 5}


In [13]:
suffix_conversion = {'ikiem': 'iczką', 'ym': 'ą', 'kim': 'ką', 'ny': 'na', 'twoim': 'twoją'}
nouns_1a = ['lekarzem', 'pisarzem', 'dziennikarzem', 'kucharzem', 'sekretarzem', 'malarzem', 'weterynarzem',
            'ochroniarzem', 'marynarzem', 'handlarzem', 'żeglarzem', 'taksówkarzem', 'piosenkarzem']
nouns_1a = {m_noun: re.sub(r"rzem\b", "rką", m_noun) for m_noun in nouns_1a}

In [14]:
female_nouns_1a, mask_nouns_1a = get_female_nouns(df_first_person, nouns_1a, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_1a sample:\n\n{female_nouns_1a['pol_text'].sample(3)}\n{'='*60}")

Num examples: 391
female_nouns_1a sample:

294                    Jestem lekarką.
118                Nie jestem lekarką.
46     Jestem lekarką, nie detektywem.
Name: pol_text, dtype: str


In [15]:
nouns_1b = {'szczęściarzem': 'szczęściarą', 'gospodarzem': 'gospodynią', 'nudziarzem': 'nudziarą'}
female_nouns_1b, mask_nouns_1b = get_female_nouns(df_first_person, nouns_1b, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_1b sample:\n\n{female_nouns_1b['pol_text'].sample(3)}\n{'='*60}")

Num examples: 40
female_nouns_1b sample:

19    Och, proszę, wybacz mi, jestem bardzo złą gosp...
34                                   Jestem gospodynią.
24                         Ja chyba jestem szczęściarą.
Name: pol_text, dtype: str


In [16]:
df_female = pd.concat([df_female, female_nouns_1a, female_nouns_1b], ignore_index=True)
df_male = pd.concat([df_male, df_first_person[mask_nouns_1b | mask_nouns_1a]], ignore_index=True)
df_first_person = df_first_person[~(mask_nouns_1b | mask_nouns_1a)].reset_index(drop=True)

- ### **2.** ...mistrzem --> ...mistrzynią (e.g. [bur]mistrzem - [bur]mistrzynią)

In [17]:
pattern_2 = r'\bjestem\b\s+\b(\w*mistrzem)\b'
nouns_male = df_first_person['pol_text'].str.extract(pattern_2, flags=re.IGNORECASE)[0].str.lower().value_counts()
nouns_male = nouns_male[nouns_male >= 5].to_dict()
print(nouns_male)

{'mistrzem': 33, 'burmistrzem': 8}


In [18]:
nouns_2 = {'mistrzem': 'mistrzynią', 'burmistrzem': 'burmistrzynią'}
suffix_conversion = {'nym': 'ną', 'kim': 'ką', 'twoim': 'twoją', 'łym': 'łą', 'tym': 'tą', 'sam': 'sama', 'szym': 'szą', 'wym': 'wą'}
female_nouns_2, mask_nouns_2 = get_female_nouns(df_first_person, nouns_2, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_2 sample:\n\n{female_nouns_2['pol_text'].sample(3)}\n{'='*60}")

Num examples: 64
female_nouns_2 sample:

19                Jestem mistrzynią tańca.
18    Jestem trzykrotną mistrzynią świata.
15                   Jestem burmistrzynią.
Name: pol_text, dtype: str


In [19]:
df_female = pd.concat([df_female, female_nouns_2], ignore_index=True)
df_male = pd.concat([df_male, df_first_person[mask_nouns_2]], ignore_index=True)
df_first_person = df_first_person[~(mask_nouns_2)].reset_index(drop=True)

- ### **3.** ...nikiem --> ...niczką/cą (e.g. ochotnikiem - ochotniczką | nieudacznikiem - nieudacznicą)

In [20]:
pattern_3 = r'\bjestem\b.+\b(\w+ikiem)\b'
nouns_male = df_first_person['pol_text'].str.extract(pattern_3, flags=re.IGNORECASE)[0].str.lower().value_counts()
nouns_male = nouns_male[nouns_male >= 4].to_dict()
print(nouns_male)

{'prawnikiem': 65, 'niewolnikiem': 32, 'dłużnikiem': 29, 'wojownikiem': 22, 'anglikiem': 21, 'mechanikiem': 20, 'pracownikiem': 18, 'kierownikiem': 17, 'urzędnikiem': 16, 'wspólnikiem': 15, 'alkoholikiem': 14, 'katolikiem': 12, 'rolnikiem': 11, 'porucznikiem': 11, 'przewodnikiem': 11, 'pułkownikiem': 10, 'magikiem': 10, 'nieudacznikiem': 10, 'komikiem': 9, 'pomocnikiem': 9, 'przeciwnikiem': 9, 'strażnikiem': 8, 'zwolennikiem': 8, 'samotnikiem': 8, 'wysłannikiem': 7, 'naczelnikiem': 7, 'grzesznikiem': 7, 'chemikiem': 6, 'zwierzchnikiem': 6, 'pośrednikiem': 6, 'rzeźnikiem': 5, 'robotnikiem': 5, 'miłośnikiem': 5, 'hydraulikiem': 5, 'górnikiem': 4, 'technikiem': 4, 'ogrodnikiem': 4, 'królikiem': 4, 'paranoikiem': 4}


In [21]:
nouns_3b = ['niewolnikiem', 'anglikiem', 'pracownikiem', 'pomocnikiem', 'grzesznikiem', 'nieudacznikiem', 'robotnikiem']
nouns_3a = [k for k in nouns_male.keys() if k not in nouns_3b+['porucznikiem', 'pułkownikiem', 'królikiem']]

nouns_3b = {k: re.sub(r"nikiem\b|likiem\b", lambda x: {'nikiem': 'nicą', 'likiem': 'ielką'}[x.group(0)], k) for k in nouns_3b}
nouns_3a = {k: re.sub(r"ikiem\b", 'iczką', k) for k in nouns_3a}

In [22]:
suffix_conversion = {'oim': 'oją', 'skim': 'ską', 'okim': 'oką', 'lkim': 'lką', 'akim': 'aką', 'wym': 'wą', 'nikiem': 'niczką',
                     'glikiem': 'gielką', 'olikiem': 'oliczką', 'ym': 'ą', 'czykiem': 'ką', 'óry': 'óra', 'dny': 'dna', 'alny': 'alna'}

female_nouns_3, mask_nouns_3 = get_female_nouns(df_first_person, nouns_3a | nouns_3b, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_3 sample:\n\n{female_nouns_3['pol_text'].sample(3)}\n{'='*60}")

Num examples: 450
female_nouns_3 sample:

333                             Jestem twoją dłużniczką.
217                             Jestem też alkoholiczką.
448    Powiedz wujkowi, że jestem farmerem, nie wojow...
Name: pol_text, dtype: str


In [23]:
df_female = pd.concat([df_female, female_nouns_3], ignore_index=True)
df_male = pd.concat([df_male, df_first_person[mask_nouns_3]], ignore_index=True)
df_first_person = df_first_person[~(mask_nouns_3)].reset_index(drop=True)

- ### **4.** Final ..em Noun Conversion

In [24]:
pattern_4 = r'\bjestem\b.+\b(\w+em)\b'
nouns_male = df_first_person['pol_text'].str.extract(pattern_4, flags=re.IGNORECASE)[0].str.lower().value_counts()
nouns_male = list(nouns_male[nouns_male >= 5].to_dict())
print(nouns_male)

['człowiekiem', 'jestem', 'przyjacielem', 'ojcem', 'facetem', 'dzieckiem', 'synem', 'szefem', 'tchórzem', 'całkiem', 'żołnierzem', 'wiem', 'głupcem', 'królem', 'bratem', 'policjantem', 'mężem', 'wrażeniem', 'oficerem', 'właścicielem', 'panem', 'więźniem', 'chłopcem', 'złodziejem', 'naukowcem', 'gościem', 'obywatelem', 'ekspertem', 'typem', 'kapitanem', 'agentem', 'członkiem', 'biznesmenem', 'detektywem', 'księdzem', 'żydem', 'fanem', 'duchem', 'aktorem', 'doktorem', 'szpiegiem', 'amerykaninem', 'szeryfem', 'wrogiem', 'dżentelmenem', 'dyrektorem', 'reporterem', 'rozumiem', 'chirurgiem', 'nauczycielem', 'wariatem', 'pilotem', 'prezesem', 'razem', 'bogiem', 'kawalerem', 'chłopakiem', 'klientem', 'bohaterem', 'inżynierem', 'oszustem', 'życiem', 'potworem', 'prawem', 'geniuszem', 'adwokatem', 'studentem', 'profesorem', 'księciem', 'powrotem', 'strzelcem', 'dupkiem', 'kumplem', 'stróżem', 'wielbicielem', 'draniem', 'partnerem', 'przejazdem', 'świadkiem', 'samurajem', 'posłańcem', 'milionerem

In [25]:
with open("../local_data/gender_pronouns/1st_person/word_lists/nouns_4a_source.pkl", 'rb') as f:
    nouns_4a = pickle.load(f)

nouns_conversion_4a = {'orem': 'orką', 'erem': 'erką', 'entem': 'entką', 'antem': 'antką', 'aninem': 'anką', 'anem': 'anką',
                       'akiem': 'aczką', 'ykiem': 'yczką', 'kiem': 'kinią', 'cielem': 'cielką', 'ejem': 'ejką',
                       'owcem': 'owczynią','atem': 'atką', 'fem': 'fką', 'aczem': 'aczką', 'ogiem': 'ożką', 'rtem': 'rtką'}

nouns_4a = {k: re.sub(rf"{r'|'.join(nouns_conversion_4a)}", lambda x: nouns_conversion_4a[x.group(0)], k) for k in nouns_4a}

In [26]:
with open("../local_data/gender_pronouns/1st_person/word_lists/nouns_4b_source.pkl", 'rb') as f:
    nouns_4b = pickle.load(f)

nouns_conversion_4b = {'cielem': 'ciółką', 'telem': 'telką', 'plem': 'pelką', 'fem': 'fową', 'źniem': 'źniarką', 'czniem': 'czennicą',
                        'nem': 'nką', 'wem': 'wką', 'ydem': 'ydówką', 'adem': 'adką', 'sem': 'ską', 'tem': 'tką', 'szem': 'szką', 
                        'ńcem': 'nką', 'dzem': 'dzką', 'uzem': 'uzką', 'rzem': 'rką', 'nkiem': 'nką', 'łem': 'licą', 'akiem': 'ką',
                        'jczykiem': 'jką', 'ńczykiem': 'nką', 'dczykiem': 'dką', 'trem': 'trą', 'irem': 'irką', 'chem': 'szką', 
                        'rykiem': 'ryczką'}

nouns_4b = {k: re.sub(rf"{r'|'.join(nouns_conversion_4b)}", lambda x: nouns_conversion_4b[x.group(0)], k) for k in nouns_4b}

In [27]:
suffix_conversion = {'jakimś': 'jakąś', 'który': 'która', 'ym': 'ą', 'twoim': 'twoją',
                     'zmęczony': 'zmęczona', 'iony': 'iona', 'onny': 'onna', 'orny': 'orna', 'ijny': 'ijna',
                     'jakim': 'jaką', 'niczyim': 'niczyją', 'samotny': 'samotna', 'nikiem': 'niczką',
                     'współpracownikiem': 'współpracownicą', 'zabezpieczony': 'zabezpieczona', 'demokratyczny': 'demokratyczna',
                     'dumny ': 'dumna ', 'dumny,': 'dumna,', 'dumny.': 'dumna.', 'sporządził': 'sporządziła', 
                     'zapłacił': 'zapłaciła', 'pewien': 'pewna', 'drugim': 'drugą', 'głupim': 'głupią', 'skim': 'ską',
                     'bkim': 'bką', 'takim': 'taką', 'lkim': 'lką', 'ckim': 'cką'}

In [28]:
female_nouns_4, mask_nouns_4 = get_female_nouns(df_first_person, nouns_4a | nouns_4b, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_4 sample:\n\n{female_nouns_4['pol_text'].sample(3)}\n{'='*60}")

Num examples: 1963
female_nouns_4 sample:

686     Komendancie, nie jestem agentką Białych.
1656                  Jestem wzorową więźniarką.
768                 Myslisz, że jestem diablicą?
Name: pol_text, dtype: str


In [29]:
df_female = pd.concat([df_female, female_nouns_4], ignore_index=True)
df_male = pd.concat([df_male, df_first_person[mask_nouns_4]], ignore_index=True)
df_first_person = df_first_person[~(mask_nouns_4)].reset_index(drop=True)

### **GROUP 3A**: Jestem + immediate adjective

In [30]:
with open("../local_data/gender_pronouns/1st_person/word_lists/skip_nouns.pkl", 'rb') as f:
    skip_nouns = pickle.load(f)

edge_cases_f = apply_mask(df_first_person, r'\bjestem\b.+\b(sług|gwiazd|glin|inn|sierot|mężczyzn|patriot|maszyn|idiot|\
                                             maszyn|sędzi|tat|sob|osob|azjat|kalek|niemow|głow|porażk|ręk|gaduł)ą\b')

mask_f = apply_mask(df_first_person, r'\bjestem\b\s+\b\w+(([wnktłrmc]|sz|pi|[^y]b)a|([ią]c|(os|ie)t|[wkłjn])ą)\b') & ~edge_cases_f

edge_cases_m = apply_mask(df_first_person, rf"\bjestem\b.+\b({'|'.join(skip_nouns)})\b")
mask_m = apply_mask(df_first_person, r'\bjestem\b\s+\b\w+(y|i|ym|im|iem|em)\b') & ~edge_cases_m

print(f"Num edge case female: {edge_cases_f.sum()} | Num edge case male: {edge_cases_m.sum()}\n")
print(f"Num female: {mask_f.sum()} | Num male: {mask_m.sum()}")
df_first_person, df_female, df_male = extract(df_first_person, df_female, df_male, mask_f, mask_m)

Num edge case female: 557 | Num edge case male: 2031

Num female: 6268 | Num male: 11226


### **GROUP 3B**: Jestem + later adjective

In [31]:
with open("../local_data/gender_pronouns/1st_person/word_lists/male_nouns.pkl", 'rb') as f:
    male_nouns = pickle.load(f)

In [32]:
edge_cases_f = apply_mask(df_first_person, r'\bjestem\b.+\b(\w+[nr]y|\w+ien|gubernatora|generała|sam)\b') | \
               apply_mask(df_first_person, r'\bjestem\b.+\b(sług|gwiazd|glin|inn|sierot|mężczyzn|patriot|maszyn|idiot|\
                                             maszyn|sędzi|tat|sob|osob|azjat|kalek|niemow|głow|porażk|ręk|gaduł|tom|besti)[ąa]\b')

mask_f = apply_mask(df_first_person, r'\bjestem\b\s+\b(.+([wnktłrmc]|sz|pi|[^y]b)a)\b') & ~edge_cases_f

edge_cases_m = apply_mask(df_first_person, rf"\bjestem\b.+\b({'|'.join(skip_nouns)})\b")
mask_m = (apply_mask(df_first_person, r'\bjestem\b.+\b\w+(ien|ny|ym|wy|ty)\b') | \
         apply_mask(df_first_person, rf"\bjestem\b.+\b({'|'.join(male_nouns)})\b")) & ~edge_cases_m 
print(f"Num edge case female: {edge_cases_f.sum()} | Num edge case male: {edge_cases_m.sum()}\n")
print(f"Num female: {mask_f.sum()} | Num male: {mask_m.sum()}")
df_first_person, df_female, df_male = extract(df_first_person, df_female, df_male, mask_f, mask_m)

Num edge case female: 4900 | Num edge case male: 1890

Num female: 2364 | Num male: 4852


### **GROUP 4**: Sam/sama

In [33]:
mask_I = apply_mask(df_first_person, r'\bI\b', col='eng_text')
mask_f = (mask_I & apply_mask(df_first_person, r'\b[Ss]ama\b|\b[Ss]amą\b')) | (mask_I & apply_mask(df_first_person, r'\b[Ss]amej\b.+bie\b'))
mask_m = mask_I & apply_mask(df_first_person, r'\b[Ss]am\b|\b[Ss]amemu\b')
print(f"Num female: {mask_f.sum()} | Num male: {mask_m.sum()}")
df_first_person, df_female, df_male = extract(df_first_person, df_female, df_male, mask_f, mask_m)

Num female: 1359 | Num male: 2922


### **GROUP 5**: ..nnam/..nienem

In [34]:
mask_f = apply_mask(df_first_person, r"\b\w+nnam\b")
mask_m = apply_mask(df_first_person, r"\b\w+nienem\b")
print(f"Num female: {mask_f.sum()} | Num male: {mask_m.sum()}")
df_first_person, df_female, df_male = extract(df_first_person, df_female, df_male, mask_f, mask_m)

Num female: 1202 | Num male: 2507


### **GROUP 6**: Czuję (się) + adjectives

In [35]:
mask_czuje = apply_mask(df_first_person, r"\bczuj[eę]\b.+\bsi[ęe]\b")
mask_f = mask_czuje & apply_mask(df_first_person, r"\b\w+(na|ka|jsza|wa)\b")
mask_m = mask_czuje & apply_mask(df_first_person, r"\b\w+(ny|ki|jszy|wy|by)\b")
print(f"Num female: {mask_f.sum()} | Num male: {mask_m.sum()}")
df_first_person, df_female, df_male = extract(df_first_person, df_female, df_male, mask_f, mask_m)

Num female: 331 | Num male: 414


In [36]:
df_female.shape, df_male.shape

((68653, 2), (161399, 2))

## **5.** Label & Save Dataset

In [37]:
df_female['self_ref'] = 'F'
df_male['self_ref'] = 'M'
df_first_person['self_ref'] = 'NA'
df_text['self_ref'] = 'NA_OTHER'

final_data = pd.concat([df_female, df_male, df_first_person, df_text], ignore_index=True)
report_shapes(female=df_female, male=df_male, first_person=df_first_person, text=df_text, final=final_data)

female: 68653 | male: 161399 | first_person: 680748 | text: 2348791 | final: 3259591


In [38]:
save_dir = "../local_data/gender_pronouns/1st_person"
os.makedirs(save_dir, exist_ok=True)

with open(f"{save_dir}/data_final_1st_person.pkl", 'wb') as f:
    pickle.dump(final_data, f)